In [ ]:
# 1. Connexion au Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Création d'un dossier temporaire dans Colab
!mkdir -p /content/dataset_temporaire

# 3. Dézipper le fichier depuis le Drive vers le stockage temporaire de Colab (l'option -q le fait silencieusement)
# /!\ REMPLACEZ LE CHEMIN CI-DESSOUS PAR LE VRAI CHEMIN DE VOTRE ZIP /!\
!unzip -q "/content/drive/MyDrive/pcd/dfdc/dfdc_train_part_18.zip" -d "/content/dataset_temporaire"

print("✅ Décompression terminée !")

Mounted at /content/drive
✅ Décompression terminée !


In [ ]:
# 1. Installation de MTCNN pour la détection de visages
!pip install mtcnn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.3 MB/s eta 0:00:00


In [ ]:


# 2. Importations nécessaires
import os
import json
import cv2
import random
from google.colab import drive
from mtcnn import MTCNN
from tqdm import tqdm

# 3. Connexion à Google Drive
drive.mount('/content/drive')

# ==========================================
# PARAMÈTRES (À MODIFIER SELON VOTRE DRIVE)
# ==========================================
# Le dossier où se trouvent vos vidéos DFDC et le metadata.json
CHEMIN_DOSSIER_DFDC = '/content/dataset_temporaire/dfdc_train_part_18'

# Le dossier où les visages extraits seront sauvegardés
CHEMIN_DOSSIER_SORTIE = '/content/drive/MyDrive/pcd/dfdc/dfdc_train_part_18'

MAX_FRAMES = 15
# ==========================================

# Création des dossiers de sortie
dossier_real = os.path.join(CHEMIN_DOSSIER_SORTIE, 'real')
dossier_fake = os.path.join(CHEMIN_DOSSIER_SORTIE, 'fake')
os.makedirs(dossier_real, exist_ok=True)
os.makedirs(dossier_fake, exist_ok=True)





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Initialisation du détecteur de visages (utilisez le GPU dans Colab pour aller plus vite)
detector = MTCNN()

def process_video(video_path, output_dir, label, video_name, max_frames=15):
    """Extrait des visages aléatoires et les range dans un dossier au nom de la vidéo."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        return

    all_frame_indices = list(range(total_frames))
    random.shuffle(all_frame_indices)

    nom_dossier_video = video_name.split('.')[0]
    chemin_dossier_video = os.path.join(output_dir, nom_dossier_video)
    os.makedirs(chemin_dossier_video, exist_ok=True)

    saved_count = 0

    for frame_idx in all_frame_indices:
        if saved_count >= max_frames:
            break

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # ==========================================
        # CORRECTION ICI : SÉCURITÉ ANTI-CRASH MTCNN
        # ==========================================
        try:
            results = detector.detect_faces(frame_rgb)
        except ValueError:
            # Si MTCNN plante sur cette frame spécifique (erreur Conv2D), on l'ignore silencieusement
            continue
        except Exception:
            # Pour toute autre erreur inattendue de l'IA sur l'image
            continue
        # ==========================================

        if results:
            bounding_box = results[0]['box']
            x, y, w, h = bounding_box
            x, y = max(0, x), max(0, y)

            face_crop = frame[y:y+h, x:x+w]

            if face_crop.shape[0] > 0 and face_crop.shape[1] > 0:
                face_resized = cv2.resize(face_crop, (224, 224))

                nom_image = f"frame_{frame_idx}.jpg"
                chemin_sauvegarde = os.path.join(chemin_dossier_video, nom_image)
                cv2.imwrite(chemin_sauvegarde, face_resized)

                saved_count += 1

    cap.release()

    if saved_count < max_frames:
        print(f"⚠️ Attention : Seulement {saved_count}/{max_frames} visages trouvés dans la vidéo {video_name}")

In [ ]:

# ==========================================
# LECTURE METADATA ET ÉQUILIBRAGE
# ==========================================
chemin_metadata = os.path.join(CHEMIN_DOSSIER_DFDC, 'metadata.json')

with open(chemin_metadata, 'r') as f:
    metadata = json.load(f)

real_videos = []
fake_videos_dict = {}

for video_name, infos in metadata.items():
    if infos['label'] == 'REAL':
        real_videos.append(video_name)
    elif infos['label'] == 'FAKE':
        original = infos['original']
        if original not in fake_videos_dict:
            fake_videos_dict[original] = []
        fake_videos_dict[original].append(video_name)

videos_to_process = []

# Équilibrage : 1 Fake (aléatoire) pour 1 Real
for real_vid in real_videos:
    videos_to_process.append((real_vid, 'REAL', dossier_real))

    if real_vid in fake_videos_dict and len(fake_videos_dict[real_vid]) > 0:
        fake_vid_choisie = random.choice(fake_videos_dict[real_vid])
        videos_to_process.append((fake_vid_choisie, 'FAKE', dossier_fake))

print(f"Total de vidéos prêtes à être traitées : {len(videos_to_process)}")



Total de vidéos prêtes à être traitées : 850


In [ ]:
# ==========================================
# LANCEMENT DU TRAITEMENT (Mise à jour pour reprendre là où ça a coupé)
# ==========================================
print("Début de l'extraction des visages...")
for video_name, label, output_dir in tqdm(videos_to_process, desc="Progression"):

    # --- AJOUT POUR REPRENDRE ---
    nom_dossier_video = video_name.split('.')[0]
    chemin_dossier_video = os.path.join(output_dir, nom_dossier_video)

    # Si le dossier existe déjà ET qu'il contient déjà des images, on passe à la suivante !
    if os.path.exists(chemin_dossier_video) and len(os.listdir(chemin_dossier_video)) > 0:
        continue
    # ----------------------------

    chemin_video = os.path.join(CHEMIN_DOSSIER_DFDC, video_name)
    if os.path.exists(chemin_video):
        process_video(chemin_video, output_dir, label, video_name)
    else:
        print(f"❌ Vidéo introuvable : {video_name}")

Début de l'extraction des visages...


Progression: 100%|██████████| 850/850 [3:06:28<00:00, 13.16s/it]
